# Cross-Validation

## Overview
A single train/test split can be **lucky or unlucky**. **K-fold cross-validation** retrains and evaluates **K** times on different partitions to estimate how well the model **generalizes**.

**Topics:**
- K-fold CV (`cross_val_score`)
- **Stratified** K-fold for classification (preserves class proportions)
- `KFold` vs `StratifiedKFold` objects
- Leave-one-out (LOO) — expensive but unbiased for tiny data
- Time series: use `TimeSeriesSplit` (no random shuffle across future)

**Time:** ~2 hours

## 1. K-fold with `cross_val_score`
Default `cv=5` uses **StratifiedKFold** for classifiers and **KFold** for regressors in recent sklearn.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)
model = LogisticRegression(max_iter=1000)

scores = cross_val_score(model, X, y, cv=5)
print('5-fold scores:', scores)
print('Mean:', scores.mean(), 'Std:', scores.std())

## 2. Explicit `StratifiedKFold`
Use when you want **shuffle** and a **random_state** for reproducibility, or when passing a splitter object into `cross_val_score`.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_s = cross_val_score(model, X, y, cv=skf)
print('Stratified (shuffled) mean:', scores_s.mean(), 'std:', scores_s.std())

## 3. `KFold` on the same data
For classification, **unstratified** KFold can give folds with **no** samples of a rare class — prefer **StratifiedKFold** for imbalance.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_k = cross_val_score(model, X, y, cv=kf)
print('KFold mean:', scores_k.mean())

## 4. Leave-one-out (LOO)
**n** folds, **n** fits — only for **very small** datasets.

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_score

X_small, y_small = X[:25], y[:25]
loo = LeaveOneOut()
scores_loo = cross_val_score(model, X_small, y_small, cv=loo)
print('LOO mean accuracy (n=25):', scores_loo.mean(), '(expensive for large n!)')

## 5. Key takeaways
- **Validation / CV** estimates generalization; **test set** (held out once) gives final unbiased estimate.
- **Stratify** classification folds when classes matter.
- For **hyperparameter tuning**, use CV **inside** training data; keep a **final test** set untouched until the end.